# External Wrappers

> 3rd party wrappers for the environments including Pettingzoo, BenchMarl, and RLLIB.

In [ ]:
#| default_exp wrappers.external

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import gymnasium as gym

from gymnasium import spaces
from typing import Any

from stable_marl.envs.multigrid.base import MultiGridEnv
from stable_marl.envs.multigrid import CONFIGURATIONS

# Optional dependencies: each wrapper only needs its own library installed
try:
    from pettingzoo import ParallelEnv
except ImportError:
    ParallelEnv = object

try:
    from ray.rllib.env import MultiAgentEnv
    from ray.tune.registry import register_env
except ImportError:
    MultiAgentEnv, register_env = object, None


## PettingZoo

Using MultiGrid environments with the PettingZoo ParallelEnv API.

**Usage**

Wrap an environment instance with :class:`.PettingZooWrapper`:

    >>> import gymnasium as gym
    >>> import stable_marl.envs
    >>> env = gym.make('MultiGrid-Empty-8x8-v0', agents=2, render_mode='human')

    >>> from stable_marl.wrappers.external import PettingZooWrapper
    >>> env = PettingZooWrapper(env)

Wrap an environment class with :func:`.to_pettingzoo_env()`:

    >>> from stable_marl.envs.multigrid import EmptyEnv
    >>> from stable_marl.wrappers.external import to_pettingzoo_env
    >>> PZEnv = to_pettingzoo_env(EmptyEnv, metadata={'name': 'empty_v0'})
    >>> env = PZEnv(agents=2, render_mode='human')

Following the ParallelEnv API, once an agent has terminated it is removed from
`env.agents` and from every dict returned by `step` (the dicts of the step in which
it terminates still include it).


In [ ]:
#| export
from typing import Any, TypeVar
import numpy as np

In [ ]:
#| export
# ActionType = TypeVar("ActionType")
AgentID = int
ObsType = dict[str, Any]
ActionType = Any

In [ ]:
#| export
class PettingZooWrapper(ParallelEnv):
    """
    Wrapper for a ``MultiGridEnv`` environment that implements the
    PettingZoo ``ParallelEnv`` interface.

    Agent IDs are the agent indices (subclasses can map them, see `_agent_id`).
    Agents that terminated in an earlier step are removed from `agents` and from
    every dict returned by `step`, as required by the ParallelEnv API.
    """

    def __init__(self, env: MultiGridEnv):
        self.env = env
        self.metadata = {}

    # Agent ID <-> agent index mapping (identity; overridden by TorchRLPettingZooWrapper)
    def _agent_id(self, index: int) -> AgentID:
        return index

    def _agent_index(self, agent_id: AgentID) -> int:
        return agent_id

    def _remap(self, d: dict) -> dict:
        return {self._agent_id(i): v for i, v in d.items()}

    def is_done(self) -> bool:
        return self.env.unwrapped.is_done()

    def reset(
        self,
        seed: int | None = None,
        options: dict | None = None,
    ) -> tuple[dict[AgentID, ObsType], dict[AgentID, dict]]:
        observations, infos = self.env.reset(seed=seed, options=options)
        infos = {i: infos.get(i, {}) for i in observations} # one info dict per agent
        return self._remap(observations), self._remap(infos)

    def step(
        self, actions: dict[AgentID, ActionType]
    ) -> tuple[
        dict[AgentID, ObsType],
        dict[AgentID, float],
        dict[AgentID, bool],
        dict[AgentID, bool],
        dict[AgentID, dict[str, Any]],
    ]:
        # Only agents that are still active act and are reported
        live = [self._agent_index(agent_id) for agent_id in self.agents]
        env_actions = {
            self._agent_index(agent_id): action
            for agent_id, action in actions.items()
            if self._agent_index(agent_id) in live
        }
        observations, rewards, terminations, truncations, infos = self.env.step(env_actions)

        def keep(d: dict, cast=lambda v: v) -> dict:
            return {self._agent_id(i): cast(d[i]) for i in live}

        return (
            keep(observations),
            keep(rewards, float),
            keep(terminations, bool),
            keep(truncations, bool),
            keep(infos),
        )

    def render(self) -> None | np.ndarray | str | list:
        return self.env.render()

    def close(self) -> None:
        return self.env.close()

    @property
    def agents(self) -> list[AgentID]:
        if self.env.unwrapped.is_done():
            return []
        return [self._agent_id(agent.index) for agent in self.env.unwrapped.agents if not agent.terminated]

    @property
    def possible_agents(self) -> list[AgentID]:
        return [self._agent_id(agent.index) for agent in self.env.unwrapped.agents]

    @property
    def observation_spaces(self) -> dict[AgentID, spaces.Space]:
        return self._remap(dict(self.env.observation_space))

    @property
    def action_spaces(self) -> dict[AgentID, spaces.Space]:
        return self._remap(dict(self.env.action_space))

    @property
    def render_mode(self) -> str | None:
        return self.env.render_mode

    def observation_space(self, agent_id: AgentID) -> spaces.Space:
        return self.env.observation_space[self._agent_index(agent_id)]

    def action_space(self, agent_id: AgentID) -> spaces.Space:
        return self.env.action_space[self._agent_index(agent_id)]


In [ ]:
#| export
def to_pettingzoo_env(
    env_cls: type[MultiGridEnv],
    *wrappers: gym.Wrapper,
    metadata: dict[str, Any] = {}) -> type[ParallelEnv]:
    """
    Convert a ``MultiGridEnv`` environment class to a PettingZoo ``ParallelEnv`` class.

    Note that this is a wrapper around the environment **class**,
    not environment instances.

    Parameters
    ----------
    env_cls : type[MultiGridEnv]
        ``MultiGridEnv`` environment class
    wrappers : gym.Wrapper
        Gym wrappers to apply to the environment
    metadata : dict[str, Any]
        Environment metadata

    Returns
    -------
    pettingzoo_env_cls : type[ParallelEnv]
        PettingZoo ``ParallelEnv`` environment class
    """
    class PettingZooEnv(PettingZooWrapper):
        def __init__(self, *args, **kwargs):
            env = env_cls(*args, **kwargs)
            for wrapper in wrappers:
                env = wrapper(env)
            super().__init__(env)

    PettingZooEnv.__name__ = f"PettingZoo_{env_cls.__name__}"
    PettingZooEnv.metadata = metadata
    return PettingZooEnv


## TorchRL

In [ ]:
#| export
class TorchRLPettingZooWrapper(PettingZooWrapper):
    """
    PettingZooWrapper with string agent IDs (``'agent_0'``, ``'agent_1'``, ...),
    as required by TorchRL's ``torchrl.envs.libs.pettingzoo.PettingZooWrapper``.

    Use ``use_mask=True`` in TorchRL when agents can terminate at different times
    (e.g. ``success_termination_mode='all'``).
    """

    def _agent_id(self, index: int) -> str:
        return f"agent_{index}"

    def _agent_index(self, agent_id: str) -> int:
        return int(agent_id.split("_")[1])


## RayLIB

The following provides tools for using MultiGrid environments with
the RLlib MultiAgentEnv API.

**Usage**

Register the environment configurations from :mod:`stable_marl.envs` with RLlib, and use one by name:

    >>> from stable_marl.wrappers.external import register_rllib_envs
    >>> register_rllib_envs()
    >>> from ray.rllib.algorithms.ppo import PPOConfig
    >>> algorithm_config = PPOConfig().environment(env='MultiGrid-Empty-8x8-v0', env_config={'agents': 2})

Wrap an environment instance with :class:`.RLlibWrapper`:

    >>> import gymnasium as gym
    >>> import stable_marl.envs
    >>> env = gym.make('MultiGrid-Empty-8x8-v0', agents=2)

    >>> from stable_marl.wrappers.external import RLlibWrapper
    >>> env = RLlibWrapper(env)

Wrap an environment class with :func:`.to_rllib_env()`:

    >>> from stable_marl.envs.multigrid import EmptyEnv
    >>> from stable_marl.wrappers.external import to_rllib_env
    >>> MyEnv = to_rllib_env(EmptyEnv, default_config={'size': 8})
    >>> env = MyEnv({'agents': 2})

**Observations**: RLlib's default models do not support the text ``mission`` or
mixed ``Dict`` observations, so by default the ``image`` and ``direction`` entries
are flattened into a single float vector (``obs_keys``, ``flatten`` arguments).
For pixel inputs use ``obs_keys=('pov',), flatten=False`` with a CNN model.


In [ ]:
#| export
class RLlibWrapper(MultiAgentEnv):
    """
    Wrapper for a ``MultiGridEnv`` environment that implements the
    RLlib ``MultiAgentEnv`` interface.

    Agents that terminated in an earlier step are removed from `agents` and from
    every dict returned by `step`, as required by RLlib.
    """

    def __init__(
        self,
        env: MultiGridEnv,
        obs_keys: tuple[str, ...] = ('image', 'direction'),
        flatten: bool = True):
        """
        Parameters
        ----------
        env : MultiGridEnv
            Environment to wrap
        obs_keys : tuple[str, ...]
            Observation entries to keep ('image', 'pov', 'direction', 'position');
            the text 'mission' is not supported by RLlib
        flatten : bool
            Concatenate the kept entries into one float32 vector (Box entries flattened,
            Discrete entries one-hot), which RLlib's default models accept.
            Otherwise a single entry is returned as is, and several as a Dict.
        """
        super().__init__()
        assert 'mission' not in obs_keys, "RLlib does not support the text 'mission' observation"
        self.env = env
        self.obs_keys = tuple(obs_keys)
        self.flatten = flatten

        base_agents = env.unwrapped.agents
        self._raw_obs_spaces = {agent.index: agent.observation_space for agent in base_agents}
        self.possible_agents = [agent.index for agent in base_agents]
        self.agents = list(self.possible_agents)
        self.observation_spaces = {i: self._convert_space(s) for i, s in self._raw_obs_spaces.items()}
        self.action_spaces = {agent.index: agent.action_space for agent in base_agents}
        self.observation_space = spaces.Dict(self.observation_spaces)
        self.action_space = spaces.Dict(self.action_spaces)

    def _convert_space(self, space: spaces.Dict) -> spaces.Space:
        parts = [space[key] for key in self.obs_keys]
        if not self.flatten:
            return parts[0] if len(parts) == 1 else spaces.Dict(dict(zip(self.obs_keys, parts)))

        lows, highs = [], []
        for part in parts:
            if isinstance(part, spaces.Discrete):
                lows.append(np.zeros(part.n, dtype=np.float32))
                highs.append(np.ones(part.n, dtype=np.float32))
            else:
                lows.append(np.asarray(part.low, dtype=np.float32).ravel())
                highs.append(np.asarray(part.high, dtype=np.float32).ravel())
        return spaces.Box(np.concatenate(lows), np.concatenate(highs), dtype=np.float32)

    def _convert_obs(self, index: int, obs: dict) -> Any:
        if not self.flatten:
            if len(self.obs_keys) == 1:
                return obs[self.obs_keys[0]]
            return {key: obs[key] for key in self.obs_keys}

        vector = []
        for key in self.obs_keys:
            space = self._raw_obs_spaces[index][key]
            if isinstance(space, spaces.Discrete):
                one_hot = np.zeros(space.n, dtype=np.float32)
                one_hot[int(obs[key])] = 1
                vector.append(one_hot)
            else:
                vector.append(np.asarray(obs[key], dtype=np.float32).ravel())
        return np.concatenate(vector)

    def reset(self, *, seed: int | None = None, options: dict | None = None):
        observations, infos = self.env.reset(seed=seed, options=options)
        self.agents = list(self.possible_agents)
        return (
            {i: self._convert_obs(i, observations[i]) for i in self.agents},
            {i: infos.get(i, {}) for i in self.agents},
        )

    def step(self, action_dict: dict[AgentID, ActionType]):
        # Only agents that are still active act and are reported
        live = list(self.agents)
        actions = {i: action for i, action in action_dict.items() if i in live}
        observations, rewards, terminations, truncations, infos = self.env.step(actions)

        obs = {i: self._convert_obs(i, observations[i]) for i in live}
        rewards = {i: float(rewards[i]) for i in live}
        terminateds = {i: bool(terminations[i]) for i in live}
        truncateds = {i: bool(truncations[i]) for i in live}
        infos = {i: infos[i] for i in live}

        terminateds['__all__'] = all(terminateds[i] for i in live)
        truncateds['__all__'] = all(truncateds[i] for i in live)
        self.agents = [i for i in live if not (terminateds[i] or truncateds[i])]
        return obs, rewards, terminateds, truncateds, infos

    def render(self):
        return self.env.render()

    def close(self):
        return self.env.close()

    def get_observation_space(self, agent_index: int) -> spaces.Space:
        return self.observation_spaces[agent_index]

    def get_action_space(self, agent_index: int) -> spaces.Space:
        return self.action_spaces[agent_index]


In [ ]:
#| export
def to_rllib_env(
    env_cls: type[MultiGridEnv],
    *wrappers: gym.Wrapper,
    default_config: dict = {},
    **rllib_kwargs) -> type[MultiAgentEnv]:
    """
    Convert a ``MultiGridEnv`` environment class to an RLLib ``MultiAgentEnv`` class.

    Note that this is a wrapper around the environment **class**,
    not environment instances.

    Parameters
    ----------
    env_cls : type[MultiGridEnv]
        ``MultiGridEnv`` environment class
    wrappers : gym.Wrapper
        Gym wrappers to apply to the environment
    default_config : dict
        Default configuration for the environment
    rllib_kwargs
        Extra arguments for :class:`.RLlibWrapper` (``obs_keys``, ``flatten``)

    Returns
    -------
    rllib_env_cls : type[MultiAgentEnv]
        RLlib ``MultiAgentEnv`` environment class
    """
    class RLlibEnv(RLlibWrapper):
        def __init__(self, config: dict = {}):
            config = {**default_config, **config}
            env = env_cls(**config)
            for wrapper in wrappers:
                env = wrapper(env)
            super().__init__(env, **rllib_kwargs)

    RLlibEnv.__name__ = f"RLlib_{env_cls.__name__}"
    return RLlibEnv


def register_rllib_envs(**rllib_kwargs):
    """
    Register every environment configuration in :data:`stable_marl.envs.multigrid.CONFIGURATIONS`
    with RLlib, under the same name (e.g. ``'MultiGrid-Empty-8x8-v0'``).
    The RLlib ``env_config`` is passed to the environment as keyword arguments.

    Parameters
    ----------
    rllib_kwargs
        Extra arguments for :class:`.RLlibWrapper` (``obs_keys``, ``flatten``)
    """
    assert register_env is not None, "ray[rllib] is required to register RLlib environments"
    for name, (env_cls, config) in CONFIGURATIONS.items():
        register_env(name, to_rllib_env(env_cls, default_config=config, **rllib_kwargs))


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()